# Ground-pass options and defensive coverage
Exploratory heuristic, not calibrated probabilities or empirical validation. Pose z is centroid-relative: aerial scoring is unsupported. Neck-to-nose direction is not eye gaze.
Install with `python -m pip install -e '.[notebook]'` from the repository first. This notebook uses only the packaged sample and never downloads a match.
The roster and movement filter below are specific to this short match-1925299 sample, not full-match lineup inference. Centered velocities are retrospective.
Compare all teammates to feet, three explicit constant-velocity run scenarios, and three analyst-selected spaces. All are ground passes. The background is lane occlusion from the ball to each grid point, not pitch-control probability or pass-completion probability.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if (root / 'src').is_dir() and str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data.pose_loading import iter_sample
from src.data.data_loader import MATCH_1925299_ROSTER, identify_active_players, compute_frame_velocities
from src.features import DefensiveLaneOcclusion, shoulder_table, gating_report
from src.visualization.pitch_viz import plot_defensive_lanes
frames = list(iter_sample())
active = identify_active_players(frames)
model = DefensiveLaneOcclusion()
display(gating_report(iter(frames)))

In [ ]:
def defense(frame, index):
    players = [p for p in frame['player_data'] if p['player_id'] in active
               and MATCH_1925299_ROSTER[p['player_id']]['team_name'].startswith('Perth')
               and MATCH_1925299_ROSTER[p['player_id']]['position_group'] != 'Goalkeeper']
    velocities = compute_frame_velocities(frames, index)
    xy = np.array([[p['x'], p['y']] for p in players], dtype=float).reshape(-1, 2)
    vel = np.array([velocities.get(p['player_id'], np.zeros(2)) for p in players]).reshape(-1, 2)
    return players, xy, vel
frame = frames[0]
carrier_id = (frame.get('possession') or {}).get('player_id')
if carrier_id not in MATCH_1925299_ROSTER:
    raise ValueError('Choose a frame with a known carrier to define the attacking team')
attacking_team = MATCH_1925299_ROSTER[carrier_id]['team_name']
receiver_ids = [p['player_id'] for p in frame['player_data'] if p['player_id'] in active
                and p['player_id'] != carrier_id
                and MATCH_1925299_ROSTER[p['player_id']]['team_name'] == attacking_team]
run_horizon_s = 0.8  # Explicit scenario, not a prediction of intended runs or receiver arrival.
run_receivers = (965698, 560986, 795537)  # Brazete, Jelacic, Waddingham in this sample.
def passing_options(current, index):
    positions = {p['player_id']: np.array([p['x'], p['y']], dtype=float) for p in current['player_data']}
    velocities = compute_frame_velocities(frames, index)
    options = {}
    for pid in receiver_ids:
        if pid not in positions:
            raise ValueError(f'Receiver {pid} missing from frame {current["frame"]}')
        options[f"Feet: {MATCH_1925299_ROSTER[pid]['short_name']}"] = (positions[pid], False)
    for pid in run_receivers:
        if pid not in velocities:
            raise ValueError(f'Cannot construct run scenario without velocity for {pid}')
        lead = positions[pid] + run_horizon_s * velocities[pid]
        lead = np.clip(lead, [-52.5, -34.], [52.5, 34.])
        options[f"Run +{run_horizon_s:.1f}s: {MATCH_1925299_ROSTER[pid]['short_name']}"] = (lead, False)
    options.update({'Space: central channel': (np.array([25., 0.]), False),
                    'Space: wide right': (np.array([22., 24.5]), False),
                    'Space: left channel': (np.array([27.5, -14.]), False)})
    return options
targets = passing_options(frame, 0)
ball = np.array([frame['ball_data']['x'], frame['ball_data']['y']])
players, xy, vel = defense(frame, 0)
rows = []
plot_targets = {}
for name, (target, _) in targets.items():
    baseline = model.compute_frame_occlusion(ball, target, xy, vel, players, use_pose=False)
    score = model.compute_frame_occlusion(ball, target, xy, vel, players)
    rows.append({'option': len(rows) + 1, 'lane': name, 'distance_m': float(np.linalg.norm(target - ball)),
                 'isotropic': baseline, 'pose_proxy': score, 'pose_delta': score - baseline})
    plot_targets[name] = (target, score)
comparison = pd.DataFrame(rows)
display(comparison.sort_values('pose_proxy').round(3))
attribution = pd.DataFrame(model.compute_counterfactual_attribution(ball, targets, xy, vel, players))
attribution['player'] = attribution['player_id'].map(lambda pid: MATCH_1925299_ROSTER[pid]['short_name'])
display(attribution.sort_values('mean_shadow_added', ascending=False))

In [ ]:
temporal = []
for index in range(0, min(100, len(frames)), 2):
    current = frames[index]
    current_ball = np.array([current['ball_data']['x'], current['ball_data']['y']])
    current_players, current_xy, current_vel = defense(current, index)
    row = {'t_sec': (current['frame'] - frame['frame']) / 25.}
    for name, (target, _) in passing_options(current, index).items():
        row[name] = model.compute_frame_occlusion(current_ball, target, current_xy, current_vel, current_players)
    temporal.append(row)
temporal_df = pd.DataFrame(temporal)
display(temporal_df.drop(columns='t_sec').agg(['min', 'mean', 'max']).T.sort_values('mean'))
grid_x, grid_y = np.linspace(-52.5, 52.5, 43), np.linspace(-34., 34., 29)
coverage = np.array([[model.compute_frame_occlusion(ball, [x, y], xy, vel, players)
                      for x in grid_x] for y in grid_y])
carrier = next((p for p in frame['player_data'] if p['player_id'] == carrier_id), None)
carrier_xy = np.array([carrier['x'], carrier['y']]) if carrier else ball
teammates = [p for p in frame['player_data'] if p['player_id'] in receiver_ids]
teammates_xy = np.array([[p['x'], p['y']] for p in teammates], dtype=float).reshape(-1, 2)
opponent_gk = next((p for p in frame['player_data'] if p['player_id'] in active
    and MATCH_1925299_ROSTER[p['player_id']]['team_name'] != attacking_team
    and MATCH_1925299_ROSTER[p['player_id']]['position_group'] == 'Goalkeeper'), None)
goalkeeper_xy = np.array([opponent_gk['x'], opponent_gk['y']]) if opponent_gk else None
focus_lanes = ['Feet: J. Brazete', "Feet: J. O'Shea", 'Feet: K. Jelacic',
               'Run +0.8s: J. Brazete', 'Run +0.8s: K. Jelacic', 'Space: central channel']
fig = plot_defensive_lanes(carrier_xy, ball, xy, [model.extract_pose_heading(p) for p in players],
    plot_targets, defenders_names=[MATCH_1925299_ROSTER[p['player_id']]['short_name'] for p in players],
    teammates_xy=teammates_xy, teammates_names=[MATCH_1925299_ROSTER[p['player_id']]['short_name'] for p in teammates],
    goalkeeper_xy=goalkeeper_xy, coverage_grid=(grid_x, grid_y, coverage),
    defenders_detected=[p.get('is_detected', False) for p in players],
    temporal_df=temporal_df[['t_sec'] + focus_lanes], title_suffix=f" | Frame {frame['frame']}")
plt.show()

All 16 options are evaluated in the table, temporal summary and leave-one-out attribution; six selected lanes are plotted over time to avoid an unreadable chart. Feet/run targets are recomputed per frame, while analyst-selected space targets remain fixed. A run target extrapolates observed velocity by 0.8 seconds and clips to pitch bounds; it does not solve receiver/ball interception or infer intent.
The map includes all 22 sample players. The defending goalkeeper is shown for context but excluded from this outfield-only scoring model. Background grid points are alternative pass destinations, not a defender possession/control model. Heading wedges are illustrative, not calibrated reach envelopes.
A low occlusion index does not guarantee a feasible/completed pass: receiver control, offside, ball-speed choice and goalkeeper interventions are not modeled. Leave-one-out values are nonadditive and can be zero for redundant defenders. No causal scanning relationship or aerial advantage is inferred.
To regenerate the README figure from this checkout after running the cells: `fig.savefig(root / 'assets' / 'ground_pass_options.png', dpi=150, bbox_inches='tight')`.